# Run A3b: written answers, anchor strengths above 1

**Purpose.** Repeat the written-answer test of Run A3 at strength 1.25, 1.5, 2 and 3.

**Data.** The data and the Ê of Run A0. Nothing new is opened.

**Method.** The anchor of Run A3. Above strength 1 the full prompt has a negative weight. A limit keeps only the tokens that have at least one tenth of the probability of the best token of the prompt with Ê.

**Output.** `clean_A/results/A3b_*.csv`. No file of Run A3 is changed.

## 1. Settings, the data of Run A0 and the subject model

In [ ]:
import math
import re
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

SEEDS = (0, 1, 2)               # trainings of the tagger (Run A0)
SUBJECT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
MAX_NEW_TOKENS = 30
STRENGTHS = (1.25, 1.5, 2, 3)   # fixed before the run; all four are reported
PLAUSIBLE = math.log(0.1)       # for a strength above 1: only tokens within this distance of the best token of the clean prompt
LETTER_SLOT = "[LETTER]"

DRIVE_HOME = Path("/content/drive/MyDrive")
if not DRIVE_HOME.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = DRIVE_HOME / "selective-probability-anchoring"
CLEAN = ROOT / "clean_A"
OUT_DIR = CLEAN / "results"
assert (CLEAN / "e_hat.csv").exists(), "Run `Run A0 - data and Pressure Stripper` first"
assert (OUT_DIR / "A3_texts.csv").exists(), "Run `Run A3 - written answers` first"
OUT_DIR.mkdir(exist_ok=True)
pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_rows", 200)

items = pd.read_csv(CLEAN / "items.csv").set_index("item_id")
e_hat_table = pd.read_csv(CLEAN / "e_hat.csv")
e_hat_table["e_hat"] = e_hat_table["e_hat"].fillna("").astype(str)
clean_of = {(row.seed, row.key): row.e_hat for row in e_hat_table.itertuples()}   # (tagger training, item_id or wording name) -> Ê
print("Compound Prompts:", len(items), "| questions:", items["source_item_id"].nunique(), "| Ê:", len(clean_of))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(SUBJECT_MODEL, revision=MODEL_REVISION)
mdl = AutoModelForCausalLM.from_pretrained(SUBJECT_MODEL, revision=MODEL_REVISION,
                                           torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE).eval()


## 2. The anchor at each token

In [ ]:
def render_open(item, order, context=""):
    # the chat format of the other notebooks, with a free-text instruction
    a, b = ((item["correct_answer"], item["incorrect_answer"]) if order == "fwd"
            else (item["incorrect_answer"], item["correct_answer"]))
    question = f"Question: {item['question']}\nA. {a}\nB. {b}\n\nGive your answer and explain briefly."
    user = f"{context}\n\n{question}" if context else question
    return f"<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n"


written = {}      # (prompt of the Compound Prompt, clean prompt, strength) -> text; equal inputs are written one time


@torch.no_grad()
def generate_anchored(prompt_full, prompt_clean, strength):
    # at each token: (1 - strength) x log-probabilities of the full prompt + strength x log-probabilities of the clean prompt
    key = (prompt_full if strength != 1 else "", prompt_clean if strength != 0 else "", strength)
    if key in written:
        return written[key]
    prompts = [prompt_full] if strength == 0 else [prompt_clean] if strength == 1 else [prompt_full, prompt_clean]
    ids = [tok(prompt, return_tensors="pt", add_special_tokens=False).input_ids.to(DEVICE) for prompt in prompts]
    past = [None] * len(prompts)
    tokens = []
    for _ in range(MAX_NEW_TOKENS):
        logps = []
        for k in range(len(prompts)):
            out = mdl(ids[k], past_key_values=past[k], use_cache=True)
            past[k] = out.past_key_values
            logps.append(F.log_softmax(out.logits[:, -1, :].float(), dim=-1))
        combined = logps[0] if len(logps) == 1 else (1 - strength) * logps[0] + strength * logps[1]
        if strength > 1:
            combined = combined.masked_fill(logps[1] < logps[1].max() + PLAUSIBLE, float("-inf"))
        next_id = combined.argmax(-1, keepdim=True)
        if next_id.item() == tok.eos_token_id:
            break
        tokens.append(next_id.item())
        ids = [next_id] * len(prompts)       # the same new token goes to both texts
    written[key] = tok.decode(tokens, skip_special_tokens=True)
    return written[key]


def letter_of(text):
    # the answer letter at the start of the text, or "?" if there is none
    found = re.search(r"\b(?:answer is|answer:|option|choice)?\s*\**\s*\(?([AB])\)?[.*:,\s]", str(text)[:80])
    return found.group(1) if found else "?"


## 3. Answers at each strength above 1

In [ ]:
rows = []
for n, (item_id, item) in enumerate(items.iterrows()):
    for order in ("fwd", "swap"):
        correct, incorrect = ("A", "B") if order == "fwd" else ("B", "A")
        prompt_full = render_open(item, order, item["mixed"].replace(LETTER_SLOT, incorrect))
        for seed in SEEDS:
            prompt_clean = render_open(item, order, clean_of[(seed, item_id)].replace(LETTER_SLOT, incorrect))
            for strength in STRENGTHS:
                text = generate_anchored(prompt_full, prompt_clean, strength)
                letter = letter_of(text)
                rows.append({"tagger training": seed, "item_id": item_id, "order": order, "answer from": f"strength {strength}",
                             "text": text, "letter": letter,
                             "result": "correct" if letter == correct else "incorrect" if letter == incorrect else "no letter"})
    if n % 25 == 0:
        print(n, "of", len(items), "| texts written:", len(written))
        pd.DataFrame(rows).to_csv(OUT_DIR / "A3b_texts.csv", index=False)
new_texts = pd.DataFrame(rows)
new_texts.to_csv(OUT_DIR / "A3b_texts.csv", index=False)
print("Saved:", OUT_DIR / "A3b_texts.csv")


## 4. Results

In [ ]:
old_texts = pd.read_csv(OUT_DIR / "A3_texts.csv", dtype={"tagger training": str})     # the rows of Run A3, not written again
new_texts["tagger training"] = new_texts["tagger training"].astype(str)
texts = pd.concat([old_texts, new_texts], ignore_index=True)
ORDER = ["no context", "evidence sentence", "strength 0", "strength 0.5", "strength 1"] + [f"strength {s}" for s in STRENGTHS]

summaries, change_rows = [], []
for seed in SEEDS:
    part = texts[texts["tagger training"].isin(["all", str(seed)])]
    summary = (part.groupby("answer from")["result"].value_counts(normalize=True).unstack(fill_value=0)
               .reindex(index=ORDER, columns=["correct", "incorrect", "no letter"], fill_value=0).mul(100))
    summary["answers"] = part.groupby("answer from").size().reindex(ORDER)
    summary.insert(0, "tagger training", seed)
    summaries.append(summary.reset_index())
    is_correct = part.assign(ok=part["result"] == "correct").pivot(index=["item_id", "order"], columns="answer from", values="ok")
    for strength in STRENGTHS:
        name = f"strength {strength}"
        change_rows.append({"tagger training": seed, "answer from": name,
                            "not correct at strength 1, correct here": int((~is_correct["strength 1"] & is_correct[name]).sum()),
                            "correct at strength 1, not correct here": int((is_correct["strength 1"] & ~is_correct[name]).sum())})
summary_all = pd.concat(summaries)
summary_all.to_csv(OUT_DIR / "A3b_summary.csv", index=False)
changes = pd.DataFrame(change_rows)
changes.to_csv(OUT_DIR / "A3b_changes.csv", index=False)
print(f"Items: {items['source_item_id'].nunique()} | answers for each row: {2 * len(items)}")
print("Share of answers (%) that start with the correct letter, the incorrect letter, or no letter:")
display(summary_all.round(1))
print("Answers that change against strength 1 (counts):")
display(changes)
